<a href="https://colab.research.google.com/github/tuankhoin/CO3133-Deep-Learning/blob/main/Week_3_Convolutional_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3133 - Deep Learning and Its Applications

# Week 3 - Convolutional Neural Networks (CNNs)

![](https://www.ibm.com/content/adobe-cms/us/en/think/topics/image-classification/jcr:content/root/table_of_contents/body-article-8/image.coreimg.png/1763388570856/image-classification-1.png)

### From Local Filters to Modern CNN Systems

This lecture develops CNNs from the basic convolution operation to modern architectures and practical data pipelines.

By the end of the lecture, you should be able to:

- explain why convolution is useful for images;
- calculate Conv2D output shapes, parameter counts and receptive fields;
- distinguish standard, depthwise, pointwise, transposed and dilated convolution;
- explain pooling, dropout and normalization layers;
- recognize common CNN building blocks and architecture families;
- choose and fine-tune pretrained CNNs;
- build a correct and reproducible data pipeline.

> **Notebook format:** Markdown lecture notes only. Code demonstrations can be added separately.

## Lecture Contents

1. Classic Conv2D and Standard Convolution
2. Efficient Convolutions
3. Transposed Convolution
4. Dilated / Atrous Convolution
5. Normalization Layers
6. Pooling and Dropout
7. CNN Building Blocks
8. CNN Architectures and Pretrained Models
9. Data Pipeline and Augmentation

---
# 1. Classic Conv2D and Standard Convolution
---

## Why Convolution?

<img src="https://static.wikia.nocookie.net/characters/images/9/99/Nemo-Seagulls_.jpg/revision/latest?cb=20171121015524" height=300 />

CNNs introduce useful **inductive biases** for image-like data:

- **Locality:** nearby pixels are strongly related.
- **Weight sharing:** the same filter is applied at every spatial location.
- **Translation equivariance:** shifting the input shifts the resulting feature map.

Compared with a fully connected layer, convolution exploits **spatial structure** instead of learning a separate weight for every pixel location.

## Conv2D: Core Idea

![](https://upload.wikimedia.org/wikipedia/commons/9/90/CNN-filter-animation-1.gif)

For an input with $C_{in}$ channels and output channel $k$:

$$
\boxed{
Y_k
=
\sum_{c=1}^{C_{in}}
X_c * W_{k,c}
+b_k,
}
$$

where $*$ denotes the convolution operation used by most deep-learning libraries: $\boxed{(f \ast g)[n] := \sum_{m=-\infty}^{\infty} f[m] g[n-m]}$

The steps:

1. extract a local patch $X_c$;
2. multiply it with a learned kernel $W_{k,c}$;
3. sum across spatial positions and input channels;
4. add a bias $b_k$;
5. reuse the same kernel everywhere else.

> A Conv2D filter behaves like a **local linear model with shared parameters**.


### Multiple filters

Each filter produces one output feature map.

Therefore, number of output channels is determined by how many filters are there: $
C_{out}=\text{number of filters}.
$

## Batching and Tensor Shapes

PyTorch commonly uses the `NCHW` ordering convention: Number of elements → Number of channels → Height → Width

$$
X\in\mathbb{R}^{B\times C_{in}\times H_{in}\times W_{in}} ⟶
Y\in\mathbb{R}^{B\times C_{out}\times H_{out}\times W_{out}}.
$$

## Conv2D Hyperparameters

| Hyperparameter | Meaning | Main effect |
|---|---|---|
| $C_{in}$ | Input channels | Depth of each filter |
| $C_{out}$ | Number of filters | Number of output channels |
| Kernel size $k$ | Spatial filter size | Local field of view |
| Stride $s$ | Step between locations | Controls spatial downsampling |
| Padding $p$ | Border extension | Controls output size / boundary behavior |
| Dilation $d$ | Spacing between kernel taps | Enlarges effective field of view |
| Bias | One learned offset per output channel | Adds $C_{out}$ parameters |

Two frequent design choices:

- $k=3,s=1,p=1$ → preserve spatial size.
- $k=3,s=2,p=1$ → approximately halve spatial size.

## Conv2D Output Shape

<img src="https://thiernobarry.net/blog/2025-02-11-conv-output-size/cnn_i5_p0_s2_dr1_k3_last_element_anotated.gif" height=300 />

For dilation $d=1$:

$$
\boxed{
H_{out}
=
\left[
\frac{H_{in}+2p-k}{s}
\right]+1
}$$

$$
\boxed{
W_{out}
=
\left[
\frac{W_{in}+2p-k}{s}
\right]+1
}$$

General form with dilation:

$$
\boxed{
H_{out}
=
\left[
\frac{H_{in}+2p-d(k-1)-1}{s}
\right]+1}
$$

The same formula applies to width.

## Conv2D Parameter Count

Number of parameters == Number of weights
- For a $k_h\times k_w$ convolution: $
\boxed{n_{params}
=
C_{out}C_{in}k_hk_w
}$
- With bias: $
\boxed{
n_{parameters}
=
C_{out}(C_{in}k_hk_w+1)
}$

Important:
- In most cases, $k_h = k_w$
- parameter count depends on **channels and kernel size**;
- it does **not** depend on image height/width;
- it does **not** depend on stride.

## Receptive Field

<img src="https://ars.els-cdn.com/content/image/1-s2.0-S2046043023001028-gr3.jpg" height=300 />


The **receptive field** of a feature is the region of the original input that can influence that feature.

For one $k × k$ convolution: $
\boxed{RF=k}$

For stacked layers, receptive field grows progressively.

A useful recurrence is

$$
\boxed{r_l = r_{l-1} + (k_l-1)j_{l-1}\\ j_l=j_{l-1}s_l}
$$

where:

- $r_l$ = receptive field size at layer $l$;
- $j_l$ = spacing or **jump** between adjacent receptive fields;
- $s_l$ = stride.

Example: two $3\times3$ convolutions with stride 1 give a $5\times5$ effective receptive field.

## Classic Conv2D: Key Takeaways

- Convolution extracts **local features**.
- Weight sharing greatly reduces parameter count compared with dense layers.
- Multiple filters learn different feature detectors.
- Deeper layers gradually obtain larger receptive fields.
- Spatial resolution is controlled mainly by stride and padding.

> **Conv2D = local filtering + channel aggregation + weight sharing.**

---
# 2. Efficient Convolutions
---

Standard Conv2D performs two things simultaneously:

1. **spatial filtering**;
2. **channel mixing**.

This is expressive but expensive because the number of weights is
$
C_{out}C_{in}k^2
$

Efficient convolutions ask:

> Can spatial filtering and channel mixing be separated?

## Depthwise Convolution

<img src="https://ars.els-cdn.com/content/image/1-s2.0-S0168169918318696-gr1.jpg" height=300 />

Depthwise convolution applies **one spatial filter independently to each input channel**.

For each channel $c$: $
\boxed{Y_c=X_c\star W_c+b_c}
$

Properties:

- no channel mixing;
- usually preserves the number of channels;
- has the same spatial receptive field as a standard convolution with the same $k,s,d$.

Parameter count with bias:

$$
\boxed{
C_{in}k^2+C_{in}
}
$$

## Pointwise Convolution

A **pointwise convolution** is a $1\times1$ convolution.

At spatial position $(i,j)$: $\boxed{
Y_k(i,j)
=
\sum_{c=1}^{C_{in}}
X_c(i,j)W_{k,c}+b_k
}$

Properties:

- mixes information **across channels**;
- does not look at neighboring pixels;
- can increase or reduce the channel dimension;
- preserves $H\times W$ when stride is 1.

Parameter count:

$$
\boxed{
C_{out}C_{in}+C_{out}
}
$$

## Depthwise Separable Convolution

<img src="https://editor.analyticsvidhya.com/uploads/45412spatial_seperable_convolution.png" height=300 />

Separate the standard convolution into two stages:
$
\text{Depthwise Conv}
\rightarrow
\text{Pointwise Conv}$

Shape progression:

$$
B\times C_{in}\times H\times W
\rightarrow
B\times C_{in}\times H'\times W'
\rightarrow
B\times C_{out}\times H'\times W'.
$$

Parameter count:

$$\boxed{
C_{in}k^2+C_{in}
+
C_{out}C_{in}+C_{out}.
}$$

Ignoring biases, the cost changes from
$
\boxed{C_{out}C_{in}k^2}$
to
 $\boxed{
C_{in}k^2+C_{in}C_{out}.
 }$

<div style="background-color: #f9f9f9; padding: 15px; border-left: 5px solid #4CAF50; border-radius: 4px;">

>### 3×3 Convolution Example
>
>**Setup:**
>* Kernel Size ($D_K$): 3 × 3
>* Input Channels ($C_{\text{in}}$): 64
>* Output Channels ($C_{\text{out}}$): 128
>
>Standard Convolution:
>* **Formula:** $D_K \times D_K \times C_{\text{in}} \times C_{\text{out}}$
>* **Calculation:** $3 \times 3 \times 64 \times 128 = \mathbf{73,728}$ parameters
>
>Depthwise Separable Convolution:
>* **Depthwise Step:** $3 \times 3 \times 64 = 576$ parameters
>* **Pointwise Step:** $1 \times 1 \times 64 \times 128 = 8,192$ parameters
>* **Total:** $576 + 8,192 = \mathbf{8,768}$ parameters
>
>**Savings:** **~88.1% parameter reduction**

</div>

## Why Depthwise Separable Convolution Is Efficient

Ignoring bias, the ratio to standard convolution is approximately

$$
\frac{C_{in}k^2+C_{in}C_{out}}
{C_{in}C_{out}k^2}
=
\frac{1}{C_{out}}+
\frac{1}{k^2}.
$$

For common settings such as $k=3$ and moderately large $C_{out}$, this can be much cheaper than standard convolution.

Trade-off:

- **Benefit:** fewer parameters and FLOPs.
- **Cost:** spatial filtering and channel interaction are factorized, so expressiveness per layer can be lower.

This is the mechanism behind MobileNet.

## Standard vs Efficient Convolution

| Operation | Spatial filtering | Channel mixing | Parameters, ignoring bias |
|---|---:|---:|---:|
| Standard Conv | Yes | Yes | $C_{in}C_{out}k^2$ |
| Depthwise Conv | Yes | No | $C_{in}k^2$ |
| Pointwise Conv | No | Yes | $C_{in}C_{out}$ |
| Depthwise Separable | Yes | Yes, separately | $C_{in}k^2+C_{in}C_{out}$ |

The design principle is **factorization**: perform expensive operations only where they are needed.

---
# 3. Transposed Convolution
---

Rather than reducing the size, this time increase the size!

<img src="https://cms.aivietnam.edu.vn/uploads/figure_16_31f56f183c.png" height=300 />

## Why Upsampling?

Dense prediction tasks such as segmentation need an output at many spatial positions.

An encoder often reduces spatial size:

$$
H\times W
\rightarrow
\frac{H}{2}\times\frac{W}{2}
\rightarrow
\cdots
$$

while increasing semantic abstraction.

A decoder must then recover higher spatial resolution.

Common choices:

- nearest/bilinear interpolation + Conv2D;
- **transposed convolution**;
- combinations with skip connections, as in U-Net-style designs.

## Transposed Conv2D: Intuition

Transposed convolution is a **learnable spatial expansion + channel transformation**.

It is closely related to the transpose/adjoint of the linear operator implemented by a standard convolution.

Important distinction:

> It is **not** the mathematical inverse of convolution and does not reconstruct information that was already lost.

A stride greater than 1 normally enlarges the feature map.

## Transposed Convolution Output Size

For one spatial dimension:
$\boxed{
H_{out}
=
(H_{in}-1)s
-2p
+d(k-1)
+o
+1}
$

where:

- $k$ = kernel size;
- $s$ = stride;
- $p$ = padding;
- $d$ = dilation;
- $o$ = output padding.

For $d=1$:
$ \boxed{
H_{out}
=
(H_{in}-1)s-2p+k+o
}$

`output_padding` resolves some shape ambiguities; it is not ordinary image padding.

## Decoder Design: Transposed Conv vs Interpolation

| Method | Learnable upsampling? | Typical characteristic |
|---|---:|---|
| Nearest + Conv | Conv only | Simple, inexpensive |
| Bilinear + Conv | Conv only | Smooth resizing followed by learned refinement |
| ConvTranspose2D | Yes | Learns the spatial expansion directly |

Transposed convolution can create **checkerboard artifacts** when kernel overlap is uneven.

A common practical alternative is:

$$
\text{Bilinear Upsample}
\rightarrow
\text{Conv2D}.
$$

For segmentation, skip connections can restore fine spatial details lost in the encoder.

---
# 4. Dilated / Atrous Convolution
---

<img src="https://images.viblo.asia/e4d46e03-a337-4853-b717-d4681da66741.gif" height=300 />

## Motivation: More Context Without Shrinking the Map

Stride and pooling enlarge effective context but reduce spatial resolution.

Dense prediction often wants both:

- large contextual receptive field;
- fine spatial resolution.

**Dilated convolution** spreads kernel taps apart instead of downsampling the output grid.

Also called **atrous convolution**.

## Dilated Convolution

For kernel size $k$ and dilation $d$, taps are spaced by $d$ input positions.

For example, with $k=3$:

- $d=1$: use positions $t,t+1,t+2$;
- $d=2$: use positions $t,t+2,t+4$.

The effective kernel size is

$$
\boxed{
k_{eff}=(k-1)d+1
}
$$

Thus a $3\times3$ kernel with dilation 2 covers the span of a $5\times5$ kernel while still using only 9 learned weights per input-output channel pair.

## Dilated Conv Output Size

For one spatial axis:

$$\boxed{
H_{out}
=
\left\lfloor
\frac{H_{in}+2p-d(k-1)-1}{s}
\right\rfloor+1}
$$

With suitable padding and $s=1$, dilation can enlarge the receptive field while keeping approximately the same spatial resolution.

Compare:

- **Stride $>1$:** subsamples the output grid.
- **Pooling:** also shrinks the spatial map.
- **Dilation $>1$:** expands the field of view without necessarily shrinking the map.

## Multi-scale Context and Gridding

Using several dilation rates provides multiple context scales.

This idea appears in **Atrous Spatial Pyramid Pooling (ASPP)** and DeepLab-style segmentation systems.

Potential problem: **gridding artifacts**.

If large dilation factors are stacked carelessly, nearby input positions may never interact, producing sparse sampling patterns.

Possible mitigations:

- mix different dilation rates;
- combine dilated and standard convolution;
- use multi-scale branches;
- avoid overly aggressive dilation schedules.

---
# 5. Normalization Layers
---

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20250217165044535522/Layer-Normalization.webp" height=300 />

## Why Normalize Feature Maps?

Deep networks may produce activations whose scale and distribution vary strongly across layers.

Normalization layers re-center and re-scale activations using statistics computed over selected axes.

For a selected set $S$:

$$
\boxed{
\mu
=
\frac{1}{|S|}
\sum_{x\in S}x\;\;\;\;\;\;\;\;\;\;\;\;
\sigma^2
=
\frac{1}{|S|}
\sum_{x\in S}(x-\mu)^2}
$$

$$\boxed{
\hat x
=
\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}
}
$$

followed optionally by

$$\boxed{
y=\gamma\hat x+\beta.
}$$

> The main difference between normalization methods is **which axes define $S$**.

## Normalization Axes for CNN Feature Maps

For
$
X\in\mathbb{R}^{B\times C\times H\times W}
$

| Layer | Statistics computed over | Batch dependent? | Typical parameters |
|---|---|---:|---:|
| BatchNorm2d | $B,H,W$ for each channel | Yes | $2C$ |
| InstanceNorm2d | $H,W$ for each sample/channel | No | $2C$ if affine |
| LayerNorm | Selected feature dimensions per sample | No | Depends on normalized shape |
| GroupNorm | $H,W$ and channels within each group, per sample | No | $2C$ |

All normally preserve tensor shape.

## Batch Normalization

For each channel $c$, BatchNorm computes statistics over the mini-batch and spatial locations:

$$\boxed{
\hat x_{b,c,h,w}
=
\frac{x_{b,c,h,w}-\mu_c}
{\sqrt{\sigma_c^2+\epsilon}}\\
y_{b,c,h,w}
=
\gamma_c\hat x_{b,c,h,w}+\beta_c.
}$$

### Training

- uses current mini-batch statistics;
- updates running mean and variance.

### Evaluation

- uses stored running statistics.

Therefore `train()` and `eval()` modes matter for BatchNorm.

## Instance, Layer and Group Normalization

<img src="https://user-images.githubusercontent.com/76557164/176494819-d7be7cd7-a181-417b-aeee-b197c86a2a18.png" height=200 />

### InstanceNorm

Normalize each sample and each channel independently across $H,W$.

Common in:

- style transfer;
- image generation;
- appearance normalization.

### LayerNorm

Normalize selected feature dimensions within each sample.

Common in:

- Transformers;
- token-based architectures.

### GroupNorm

Divide channels into groups and normalize each group independently per sample.

Useful when batch sizes are small, especially in detection and segmentation.

## Choosing a Normalization Layer

| Situation | Typical choice |
|---|---|
| CNN with reasonably large stable batches | BatchNorm |
| Small batch size | GroupNorm |
| Transformer / token model | LayerNorm |
| Style / appearance-oriented image tasks | InstanceNorm |

Normalization is part of architecture design, not just preprocessing.

A common block is

$$
\text{Conv}
\rightarrow
\text{Norm}
\rightarrow
\text{Activation}.
$$

---
# 6. Pooling and Dropout
---

Pooling and Dropout solve different problems:

- **Pooling:** spatial summarization / downsampling.
- **Dropout:** regularization.

Pooling usually changes feature-map geometry; Dropout usually preserves it.

## Pooling

<img src="https://journals.plos.org/plosone/article/figure/image?size=medium&id=10.1371/journal.pone.0276523.g004" height=200 />

Pooling applies a fixed local operation independently to each channel:

$$
Y_{b,c,i,j}
=
\operatorname{pool}
(X_{b,c,\text{local window}})
$$

- Max pooling: Keeps the strongest local response.
- Average pooling: Summarizes average local evidence.

Pooling has **zero learnable parameters**.

## Pooling Output Shape and Global Average Pooling

For kernel size $k$, stride $s$, padding $p$:
$\boxed{
H_{out}
=
\left\lfloor
\frac{H+2p-k}{s}
\right\rfloor+1}
$

Channels are preserved.

### Global Average Pooling (GAP)

Average over the entire spatial dimension:

$$\boxed{
Y_{b,c}
=
\frac{1}{HW}
\sum_{h=1}^{H}
\sum_{w=1}^{W}
X_{b,c,h,w}}
$$

This converts

$$
B\times C\times H\times W
\rightarrow
B\times C.
$$

Modern classification CNNs often use GAP before the classifier instead of large fully connected layers.

## Dropout

<img src="https://miro.medium.com/1*iWQzxhVlvadk6VAJjsgXgg.png" height=300 />

During training, each activation is dropped with probability $p$.

Modern frameworks usually use **inverted dropout**:
$\boxed{
Y
=
\frac{M\odot X}{1-p}}
$

where
$
M_i\sim\operatorname{Bernoulli}(1-p).
$

The scaling preserves expected activation magnitude during training.

### Evaluation

Dropout is disabled, so no extra scaling is required.

Dropout has no learnable parameters and normally preserves tensor shape.

## Dropout vs Dropout2D

- Standard Dropout - Drops individual elements:
$
X_{b,c,h,w}.
$
- Spatial Dropout / Dropout2D - Drops entire feature channels for a sample:
$
X_{b,c,:,:}.
$

For convolutional feature maps, Dropout2D can be more meaningful because nearby spatial values within a channel are strongly correlated.

Common placements:

- near classifier heads;
- after higher-level convolution blocks;
- more often when datasets are small.

Very aggressive dropout in early CNN layers can remove useful low-level structure.

## Pooling vs Dropout

| Property | Pooling | Dropout |
|---|---|---|
| Main purpose | Spatial summarization | Regularization |
| Learnable parameters | 0 | 0 |
| Changes $H,W$ | Usually | No |
| Changes channels | No | No |
| Different train/eval behavior | No | Yes |
| Effect on receptive field | Enlarges effective context downstream | No direct geometric change |

A conventional block may look like this: $
\text{Conv}
\rightarrow
\text{Norm}
\rightarrow
\text{Activation}
\rightarrow
(\text{Pool})
\rightarrow
(\text{Dropout}).
$

---
# 7. CNN Building Blocks
---

<img src="https://assets.insightmediagroup.io/media/wp-content/uploads/2021/06/1Ky6dbD8Z92Po-zVGE8gtNw.png" height=300 />

Modern CNNs are designed by composing reusable **blocks**, not by placing isolated convolution layers one by one.

The design space includes:

- basic convolution blocks;
- residual blocks;
- bottlenecks;
- depthwise separable blocks;
- inverted residuals;
- attention blocks.

CNN design is a trade-off between **representation power, optimization, memory and compute**.

## 7.1 Basic Convolution Block

A common basic block is
$
\text{Conv2D}
\rightarrow
\text{BatchNorm}
\rightarrow
\text{ReLU}.
$

<!-- Example:

For a $3\times3$ convolution:
$
B\times C_{in}\times H\times W
\rightarrow
B\times C_{out}\times H\times W
$

when padding preserves resolution.

Parameter count with bias:

$$
C_{out}(9C_{in}+1).
$$

This is a strong baseline when compute is not highly constrained. -->

## 7.2 Residual Block

<img src="https://upload.wikimedia.org/wikipedia/commons/b/ba/ResBlock.png?utm_source=en.wikipedia.org&utm_campaign=index&utm_content=original" height=200 />

Residual learning adds a shortcut:
$\boxed{
y=F(x)+x}
$

Rather than learning the full transformation directly, the block learns a residual $F(x)$.

Benefits:

- easier optimization of very deep networks;
- direct gradient paths through skip connections;
- strong default for deep classification, detection and segmentation backbones.

If shapes do not match, use a projection shortcut:
$\boxed{
y=F(x)+W_sx}$

where $W_s$ is often a $1\times1$ convolution.

## Bottleneck Block

<img src="https://blog.kakaocdn.net/dna/bCVqgq/btqTowek3fE/AAAAAAAAAAAAAAAAAAAAAObbBGfLPSCsP4VzXqwcZUi154fROyeHpFJz00ALTCbq/img.png?credential=yqXZFxpELC7KVnFOS48ylbz2pIh7yKj8&expires=1790780399&allow_ip=&allow_referer=&signature=cU9SwsqWFUs%2BBxuuSslJ0gd4kv8%3D" height=400 />

A bottleneck reduces expensive computation by using $1\times1$ convolutions around a spatial convolution:

$$
1\times1\ \text{reduce}
\rightarrow
3\times3
\rightarrow
1\times1\ \text{expand}.
$$

The expensive $3\times3$ operation runs on a smaller intermediate channel dimension.

Typical use:

- deep ResNet variants;
- architectures where compute and memory need to be controlled.

## Depthwise Separable Block

Basically a basic block that uses depthwise separable convoution:
$
\text{Depthwise }k\times k
\rightarrow
\text{Pointwise }1\times1
$

It separates:

- spatial filtering;
- channel mixing.

This greatly reduces FLOPs and parameters, making it attractive for mobile and embedded models.

Caution: efficient arithmetic does not always translate directly into lower wall-clock latency on every hardware platform.

## Inverted Residual

<img src="https://www.researchgate.net/publication/358518820/figure/fig2/AS:1157628617072649@1653011578181/Residual-block-12-36-and-inverted-residual-block.png" height=300 />

MobileNetV2-style inverted residual blocks reverse the classic bottleneck idea:

$$
\text{Expand}
\rightarrow
\text{Depthwise Conv}
\rightarrow
\text{Project}.
$$

Typical structure:

1. $1\times1$ expansion to a larger hidden channel space;
2. depthwise spatial convolution;
3. linear $1\times1$ projection back to a compact representation;
4. residual connection when shapes are compatible.

The residual path operates between **thin bottleneck representations**, while nonlinear processing occurs in an expanded hidden space.

## Attention Blocks: Squeeze-and-Excitation

<img src="https://www.hollywoodreporter.com/wp-content/uploads/2017/08/shutterstock_297886754_-_h_2017.jpg?w=1296&h=730&crop=1" height=300 />

A Squeeze-and-Excitation (SE) block learns channel-wise importance.

- Squeeze: global average pool each channel:
$\boxed{
z_c
=
\frac{1}{HW}
\sum_{h,w}X_{c,h,w}}
$

- Excitation: Transform $\mathbf{z}$ through a small learned network to obtain channel weights $\mathbf{s}$.
- Reweight: $\boxed{
\tilde X_c=s_cX_c}
$

SE therefore answers:

> Which channels should receive more or less emphasis for this input?

## CNN Block Comparison

| Block | Main purpose | Efficiency | Typical use |
|---|---|---:|---|
| Basic Conv | General feature extraction | Medium | Standard CNN stages |
| Residual | Easier deep optimization | Medium | ResNet-like backbones |
| Bottleneck | Reduce expensive computation | High | Deep residual networks |
| Depthwise Separable | Cheap spatial + channel factorization | Very high | MobileNet-like models |
| Inverted Residual | Efficient mobile representation | Very high | MobileNetV2 / efficient CNNs |
| SE / Attention | Adaptive feature reweighting | Added overhead | Accuracy-oriented blocks |

Modern architectures are mostly **recipes for arranging and scaling these block ideas**.

---
# 8. CNN Architectures and Pretrained Models
---

![](https://cdn.analyticsvidhya.com/wp-content/uploads/2020/10/90650dnn2.webp)

## Anatomy of a CNN Architecture

A modern CNN normally consists of:

- **stem:** early extraction from the raw image;
- **stages:** repeated blocks at a given resolution;
- **downsampling:** reduce $H,W$ while increasing semantic abstraction;
- **head:** convert final features into task outputs.

A common classification pattern is

$$
\text{Image}
\rightarrow
\text{Stem}
\rightarrow
\text{Stage 1}
\rightarrow
\cdots
\rightarrow
\text{Stage 4}
\rightarrow
\text{Pooling}
\rightarrow
\text{Classifier}.
$$

## CNN Feature Hierarchy

![](https://ik.imagekit.io/upgrad1/abroad-images/imageCompo/images/how_does_CNN_architecture_workUDMCW9.png)

### Early stages

- high spatial resolution;
- edges, corners, colors, textures.

### Middle stages

- object parts;
- motifs;
- local shape patterns.

### Late stages

- low spatial resolution;
- larger receptive fields;
- high-level semantic concepts.

CNNs therefore progressively trade **spatial detail** for **semantic abstraction**.

## Short CNN Architecture Timeline

| Year | Family | Main contribution |
|---:|---|---|
| 1998 | LeNet | Early CNN for digit recognition |
| 2012 | AlexNet | Deep CNN + GPU + ReLU + Dropout |
| 2014 | VGG | Deep uniform stack of $3\times3$ convs |
| 2014 | GoogLeNet / Inception | Multi-scale branches |
| 2015 | ResNet | Residual connections enable very deep nets |
| 2017 | MobileNet | Depthwise separable convolution |
| 2019 | EfficientNet | Compound scaling of depth, width and resolution |
| 2022 | ConvNeXt | Modernized pure convolutional architecture |

The progression moves from **deeper networks** toward **better blocks, efficiency and scaling**.

## VGG

<img src="https://miro.medium.com/1*B_ZaaaBg2njhp8SThjCufA.png" height=300 />

Core idea:

$$
3\times3
\rightarrow
3\times3
\rightarrow
\text{Pool}
$$

repeated across stages.

Strengths:

- simple and uniform;
- easy to understand;
- useful educational baseline.

Limitations:

- many parameters;
- expensive computation;
- large fully connected classifier in classical versions.

## ResNet

<img src="https://miro.medium.com/1*BnoNVpj7uCNMOFOj1DQBQA.png" height=300 />

Core idea:

$$
y=F(x)+x.
$$

Residual blocks make much deeper networks easier to optimize.

Common stage pattern:

- repeat residual blocks at fixed resolution;
- downsample at stage boundaries;
- increase channel width as spatial size decreases.

Strengths:

- reliable general-purpose backbone;
- strong pretrained ecosystem;
- widely used in classification, detection and segmentation.

## Inception / GoogLeNet

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20260507143902524885/convulation_3.webp" height=300 />

Inception processes the same input through multiple branches, for example:

- $1\times1$ convolution;
- $3\times3$ convolution;
- larger / factorized convolution;
- pooling branch.

Outputs are concatenated along the channel dimension.

Key idea:

> Let the network process information at **multiple spatial scales** in parallel.

$1\times1$ convolutions are also used to control channel cost.

## DenseNet

<img src="https://miro.medium.com/0*7H9mNwLLWFiLYhLb.jpeg" height=300 />

DenseNet connects each layer to many later layers through concatenation:

$$
x_l
=
H_l([x_0,x_1,\ldots,x_{l-1}]).
$$

Instead of repeatedly relearning similar features, later layers can directly reuse earlier feature maps.

The **growth rate** controls how many new channels each layer contributes.

Strengths:

- strong feature reuse;
- efficient gradient flow.

Trade-off:

- concatenated features can increase memory traffic and implementation complexity.

## MobileNet

<img src="https://www.researchgate.net/publication/340470168/figure/fig3/AS:877502188769280@1586224233016/llustration-of-the-MobileNet-architecture-A-The-overall-MobileNet-architecture-and-B.png" height=300 />

MobileNet targets low-compute deployment.

Main idea:

$$
\text{Depthwise Conv}
\rightarrow
\text{Pointwise Conv}.
$$

MobileNetV2 adds **inverted residuals and linear bottlenecks**.

Strengths:

- low parameter/FLOP cost;
- suitable for mobile and embedded devices.

Caution:

- real latency depends on hardware and kernel implementations, not FLOPs alone.

## EfficientNet

<img src="https://www.researchgate.net/publication/379146600/figure/fig2/AS:11431281230638339@1711040630754/EfficientNet-Architecture.jpg" height=300 />

<img src="https://iq.opengenus.org/content/images/2022/11/MBconv_se-1.png" height=200 />



EfficientNet argues that model scaling should balance three dimensions:

- depth;
- width;
- input resolution.

Instead of increasing only one dimension, **compound scaling** increases them together according to a coordinated rule.

Key message:

> Bigger models should be scaled systematically, not arbitrarily.

EfficientNet combines efficient building blocks with controlled scaling.

## ConvNeXt

<img src="https://www.researchgate.net/publication/374280827/figure/fig1/AS:11431281731651755@1763485359198/ConvNeXt-a-ConvNeXt-structure-b-ConvNeXt-block-c-down-sampling.jpg" height=500 />

ConvNeXt revisits pure CNN design using lessons from modern Transformer-era architectures.

High-level ideas include:

- larger depthwise kernels;
- inverted-bottleneck-like blocks;
- LayerNorm-style normalization choices;
- updated stage and training recipes.

Important lesson:

> Improvements often come from the **whole design recipe**, not from a single new operation.

ConvNeXt demonstrates that carefully modernized CNNs remain highly competitive.

## Choosing an Architecture

Ask:

- How accurate must the model be?
- What is the latency budget?
- What is the memory budget?
- What hardware will run inference?
- Is training compute limited?
- Do we need dense prediction or only classification?

| Constraint | Common starting direction |
|---|---|
| Strong general baseline | ResNet / ConvNeXt |
| Mobile / embedded | MobileNet / efficient lightweight CNN |
| Very simple teaching baseline | VGG-like stack |
| Multi-scale processing | Inception-style ideas |
| Memory-efficient feature reuse | DenseNet-like ideas |
| Balanced scaling | EfficientNet |

Architecture choice is a trade-off among **accuracy, latency, memory and deployment constraints**.

## Pretrained Models

Large CNNs are commonly pretrained on large datasets such as ImageNet.

Benefits:

- faster convergence;
- better performance with limited target data;
- reusable low/mid-level visual features;
- reduced training cost.

Transfer-learning workflow:

$$
\text{Pretrained Backbone}
\rightarrow
\text{Replace Head}
\rightarrow
\text{Train New Head}
\rightarrow
\text{Fine-tune Some/All Layers}.
$$

## Fine-tuning Strategy

A practical progression:

1. Load pretrained weights.
2. Use the preprocessing associated with those weights.
3. Replace the task-specific classifier/head.
4. Train the new head first if the target dataset is small.
5. Unfreeze later stages gradually if needed.
6. Fine-tune with a smaller learning rate than training from scratch.
7. Evaluate on a held-out validation/test split.

Fine-tune more aggressively when:

- the target dataset is large;
- the target domain differs strongly from pretraining.

---
# 9. Data Pipeline and Augmentation
---

A CNN system is more than the network:

$$
\boxed{
\text{Model System}
=
\text{Architecture}
+
\text{Data Pipeline}
+
\text{Optimization}
}
$$

The end-to-end pipeline is

$$
\text{Raw Data}
\rightarrow
\text{Split}
\rightarrow
\text{Dataset}
\rightarrow
\text{Transform}
\rightarrow
\text{Collate}
\rightarrow
\text{DataLoader}
\rightarrow
\text{Model}.
$$

Pipeline choices directly affect gradients, metrics and deployment behavior.

## Train / Validation / Test and Leakage

![](https://cdnphoto.dantri.com.vn/VZAaDewWDyvjDvSLSP1jbDMeLhQ=/zoom/1200_630/2026/06/09/3-1781024239931.jpg)

- **Train:** learn model parameters.
- **Validation:** tune hyperparameters and select checkpoints.
- **Test:** final evaluation only.

Never use the test set for training, tuning, early stopping or checkpoint selection.

Common leakage examples:

- adjacent frames from the same video across train/test;
- the same patient/person/object instance across splits;
- near-duplicate samples across splits;
- augmenting first and splitting later.

Leakage produces deceptively high performance.

## Distribution Mismatch

Training and deployment data may follow different distributions:

$$
X_{train}\sim P_{train}(x),
$$

$$
X_{real}\sim P_{real}(x).
$$

If

$$
P_{train}(x)\neq P_{real}(x),
$$

performance may drop substantially.

A good pipeline attempts to make training conditions representative of real deployment conditions.

## Dataset Abstraction and Label Formats

A dataset provides the sample-level contract:

- `len()` → number of samples;
- `getitem(idx)` → one sample and target.

Common target formats:

| Task | Typical target |
|---|---|
| Single-label classification | Integer class index |
| Multi-label classification | Multi-hot vector |
| Segmentation | $(H,W)$ or $(C,H,W)$ mask |
| Detection | Boxes + class labels |

Target format must match:

- model output;
- loss function;
- evaluation metric.

## Augmentation

<img src="https://www.researchgate.net/publication/394397210/figure/fig4/AS:11431281577850691@1754622567182/The-figure-shows-the-outcomes-of-applying-various-data-augmentation-techniques-to-images.png" height=300 />

Augmentation does more than create additional files—it **shapes the training distribution**.

### Geometric

- crop;
- flip;
- rotate;
- scale;
- perspective.

### Photometric

- brightness;
- contrast;
- saturation / hue;
- blur.

### Regularization-oriented

- MixUp;
- CutMix;
- Random Erasing.

Critical rule:

> An augmentation should preserve the task semantics.

## Train vs Evaluation Transforms

### Training

Use random augmentation to expose the model to plausible variation.

### Validation / Test

Use deterministic preprocessing so metrics are stable and comparable.

For structured outputs such as detection, segmentation or pose estimation, geometric transformations must also be applied consistently to:

- bounding boxes;
- masks;
- keypoints;
- polygons.

## Pretrained Model Preprocessing

Pretrained checkpoints assume a specific input pipeline, potentially including:

- image size;
- crop strategy;
- normalization mean and standard deviation;
- color channel order;
- tensor layout.

A preprocessing mismatch can significantly hurt transfer-learning performance.

> Treat the checkpoint and its preprocessing recipe as one package.

## Collate and DataLoader

### Collate

Converts a list of individual samples into a batch.

Default stacking works when shapes match.

Custom collation is needed when:

- images have variable sizes;
- targets have variable length;
- detection targets contain a variable number of boxes.

### DataLoader

Controls:

- batch size;
- shuffling;
- number of workers;
- pinned memory;
- custom collation;
- prefetching / persistent workers.

Thus it affects both **statistics** and **hardware throughput**.

## Pipeline Performance

Possible data-loader bottleneck symptoms:

- low GPU utilization;
- long waits between batches;
- spare CPU resources;
- changing model size barely changes throughput.

Possible actions:

- increase data-loader workers gradually;
- enable pinned memory for CUDA training;
- use persistent workers;
- cache expensive decode operations;
- simplify or accelerate augmentation;
- benchmark samples/second rather than guessing.

Goal:

> Keep the accelerator busy without exhausting RAM, CPU or storage bandwidth.

## Class Imbalance

<img src="https://github.com/tuankhoin/CO3057-Computer-Vision/blob/main/assets/w10.JPG?raw=true" height=300/>

Problems:

- rare classes appear less frequently;
- gradients are dominated by common classes;
- overall accuracy can hide poor minority-class recall.

Solutions:

### Sampling level

- weighted sampling;
- balanced sampling.

### Loss level

- class-weighted cross-entropy;
- focal loss.

### Evaluation level

- per-class Precision / Recall / F1;
- macro-F1;
- confusion matrix;
- minority-class performance.

## Fair Evaluation Protocol

Fair architecture comparison requires keeping the following fixed:

- data split;
- preprocessing;
- augmentation policy where appropriate;
- metric;
- checkpoint-selection rule.

Otherwise we may accidentally compare **different experimental pipelines**, not different models.

For classification, useful reports include:

- Accuracy;
- macro-F1;
- per-class Recall;
- confusion matrix.

## Reproducibility and Debugging

Record:

- random seeds;
- split indices;
- transformation/augmentation configuration;
- software versions;
- hardware environment;
- checkpoint-selection rule.

Before full training:

1. inspect batches after all transforms;
2. check $(B,C,H,W)$ and tensor ranges;
3. verify image-label alignment;
4. train on a tiny subset and confirm the model can overfit it;
5. monitor loading time and accelerator utilization.

> Debug the pipeline before blaming the architecture.

---
# Week 3 Summary
---

The complete CNN picture is now:

$$
\boxed{
\text{Data Pipeline}
\rightarrow
\text{CNN Backbone}
\rightarrow
\text{Task Head}
\rightarrow
\text{Loss / Optimization}
\rightarrow
\text{Evaluation}
}
$$

Key connections:

- **Standard Conv2D** learns local shared filters.
- **Depthwise + pointwise convolution** improves efficiency.
- **Transposed convolution** performs learnable upsampling.
- **Dilated convolution** expands context without necessarily shrinking spatial maps.
- **Normalization** stabilizes activation statistics.
- **Pooling** summarizes spatial information; **Dropout** regularizes.
- **Building blocks** package these ideas into reusable structures.
- **Architecture families** differ in how blocks, stages and scaling are organized.
- **Pretraining** makes strong CNN features reusable.
- **Data pipelines and augmentation** are first-class components of model performance.

## Week 3 Checklist

Before moving on, you should be able to answer:

1. Why does convolution use fewer parameters than a fully connected image layer?
2. How do $C_{in}$ and $C_{out}$ relate to convolution filters?
3. How do you calculate Conv2D output size and parameter count?
4. What is a receptive field, and how does it grow across layers?
5. What is the difference between standard, depthwise and pointwise convolution?
6. Why is depthwise separable convolution efficient?
7. How do transposed and dilated convolutions differ?
8. What axes do BatchNorm, InstanceNorm, LayerNorm and GroupNorm normalize?
9. What is the difference between pooling and dropout?
10. Why do residual connections help deep networks?
11. What design ideas distinguish VGG, ResNet, Inception, DenseNet, MobileNet, EfficientNet and ConvNeXt?
12. Why must pretrained models use the correct preprocessing pipeline?
13. How can augmentation improve generalization but also break a task?
14. What forms of data leakage commonly occur in computer vision datasets?
15. Why should architecture comparisons use the same data and evaluation protocol?